# 129 — GNN Behaviour Cloning analysis

Episode `78867640`, focus step **35**.

- Load `128-GNN_BC/model_epoch100.pt`
- `reachable_max_ships` filtered to my planets at step 35
- `actions_to_copy` ground truth
- Model predictions with probabilities

In [1]:
import sys
import importlib.util
import math
import numpy as np
import polars as pl
import pandas as pd
import torch
from pathlib import Path

# Load 128-GNN_BC as a module (name starts with digit, use importlib)
spec = importlib.util.spec_from_file_location('gnn_bc', '128-GNN_BC.py')
gnn_bc = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gnn_bc)
sys.modules['gnn_bc'] = gnn_bc  # torch_geometric Inspector resolves type hints via sys.modules

GNNBehaviourCloning = gnn_bc.GNNBehaviourCloning
build_graph         = gnn_bc.build_graph
load_episode        = gnn_bc.load_episode

EPISODE    = 78867640
FOCUS_STEP = 35
PRE_DIR    = Path(f'126-precompute/{EPISODE}')
MODEL_PATH = Path('128-GNN_BC/model_epoch100.pt')

In [2]:
planete, planete_step, reachable_base_2, reachable_max_ships, actions_to_copy = load_episode(PRE_DIR)

for name, df in [
    ('planete',             planete),
    ('planete_step',        planete_step),
    ('reachable_base_2',    reachable_base_2),
    ('reachable_max_ships', reachable_max_ships),
    ('actions_to_copy',     actions_to_copy),
]:
    print(f'{name:25s}  {str(df.shape):18s}  cols: {df.columns}')

planete                    (17268, 6)          cols: ['id', 'step', 'x', 'y', 'production', 'nature']
planete_step               (344988, 5)         cols: ['id', 'step', 'future_step', 'ships', 'owner']
reachable_base_2           (4172992, 5)        cols: ['id_src', 'step_src', 'id_tgt', 'step_tgt', 'ships_sent']
reachable_max_ships        (284012, 6)         cols: ['id_src', 'step_src', 'angle', 'ships_sent', 'id_tgt', 'step_tgt']
actions_to_copy            (88, 5)             cols: ['step', 'id_src', 'angle', 'ships_sent', 'id_tgt']


In [14]:
planete_step.to_pandas().query("step == @FOCUS_STEP and future_step == @FOCUS_STEP")

,id,step,future_step,ships,owner
7541,18,35,35,1,0
8920,12,35,35,2,0
35200,24,35,35,14,-1
38034,14,35,35,8,-1
74638,9,35,35,85,-1
133790,15,35,35,3,1
155931,30,35,35,28,-1
158601,7,35,35,45,-1
169836,8,35,35,85,-1
174099,21,35,35,11,1


## My planets at step 35

In [3]:
T = FOCUS_STEP

ps_now = planete_step.filter(
    (pl.col('step') == T) & (pl.col('future_step') == T)
)
my_planet_ids = ps_now.filter(pl.col('owner') == 0)['id'].to_list()
print(f'My planets at step {T}: {sorted(my_planet_ids)}')

ps_now.sort('id').to_pandas()

My planets at step 35: [12, 18, 20, 22]


,id,step,future_step,ships,owner
0,0,35,35,43,-1
1,1,35,35,43,-1
2,2,35,35,43,-1
3,3,35,35,43,-1
4,4,35,35,45,-1
5,5,35,35,45,-1
6,6,35,35,45,-1
7,7,35,35,45,-1
8,8,35,35,85,-1
9,9,35,35,85,-1


## `reachable_max_ships` — owned by me, step_src == 35

In [4]:
reachable_mine = reachable_max_ships.filter(
    (pl.col('step_src') == T) & (pl.col('id_src').is_in(my_planet_ids))
).sort(['id_src', 'id_tgt'])

print(f'{len(reachable_mine)} candidate actions from my {len(my_planet_ids)} planet(s)')
reachable_mine.to_pandas()

39 candidate actions from my 4 planet(s)


,id_src,step_src,angle,ships_sent,id_tgt,step_tgt
0,12,35,-3.003457,2,1,54
1,12,35,2.096969,2,5,49
2,12,35,0.435150,2,14,54
3,12,35,0.800887,2,18,51
4,12,35,1.207544,2,20,38
5,12,35,2.810567,2,25,49
6,12,35,0.033505,2,30,54
7,18,35,1.459978,1,0,46
8,18,35,-0.048997,1,4,54
9,18,35,-1.751253,1,14,40


## `actions_to_copy` — ground truth at step 35

In [5]:
actions_35 = actions_to_copy.filter(pl.col('step') == T)
print(f'{len(actions_35)} ground truth action(s) at step {T}')
actions_35.to_pandas()

1 ground truth action(s) at step 35


,step,id_src,angle,ships_sent,id_tgt
0,35,18,-1.4326,18,14


## Load model — epoch 100

In [6]:
model = GNNBehaviourCloning(
    H  = gnn_bc.HIDDEN_DIM,
    L1 = gnn_bc.NUM_LAYERS_P1,
    L2 = gnn_bc.NUM_LAYERS_P2,
)
state_dict = torch.load(MODEL_PATH, map_location='cpu', weights_only=True)
model.load_state_dict(state_dict)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Loaded  : {MODEL_PATH}')
print(f'Params  : {n_params:,}')

Loaded  : 128-GNN_BC\model_epoch100.pt
Params  : 142,785


## Build graph for step 35 and run inference

In [7]:
graph = build_graph(
    T,
    planete, planete_step, reachable_base_2, reachable_max_ships, actions_to_copy
)
assert graph is not None, 'build_graph returned None — no action_max nodes at this step'

n_ams = graph['action_max'].x.shape[0]
n_pos = int(graph['action_max'].y.sum().item())
print(f'planet nodes       : {graph["planet"].x.shape[0]}')
print(f'planet_step nodes  : {graph["planet_step"].x.shape[0]}')
print(f'action_max nodes   : {n_ams}  (positives={n_pos})')
print()
print(graph)

planet nodes       : 32
planet_step nodes  : 672
action_max nodes   : 39  (positives=1)

HeteroData(
  planet={ x=[32, 4] },
  planet_step={ x=[672, 9] },
  action_max={
    x=[39, 1],
    y=[39],
  },
  (planet, has_ps, planet_step)={ edge_index=[2, 672] },
  (planet_step, rev_has_ps, planet)={ edge_index=[2, 672] },
  (planet_step, reaches, planet_step)={
    edge_index=[2, 23859],
    edge_attr=[23859],
  },
  (planet_step, src_of, action_max)={ edge_index=[2, 39] },
  (action_max, rev_src_of, planet_step)={ edge_index=[2, 39] },
  (planet_step, tgt_of, action_max)={ edge_index=[2, 39] },
  (action_max, rev_tgt_of, planet_step)={ edge_index=[2, 39] }
)


In [8]:
with torch.no_grad():
    logits = model(graph)          # (n_ams,)
    probs  = torch.sigmoid(logits).numpy()

print(f'logits : [{logits.min():.3f}, {logits.max():.3f}]')
print(f'probs  : [{probs.min():.3f}, {probs.max():.3f}]')

logits : [-5.605, 4.764]
probs  : [0.004, 0.992]


## Predicted actions vs ground truth

Reconstruct which `(id_src, id_tgt)` corresponds to each `action_max` node
by replicating the validity filter from `build_graph`.

In [9]:
# ── Rebuild PS key index ──────────────────────────────────────────────────────
ps_raw          = planete_step.filter(pl.col('step') == T)
ps_ids_np       = ps_raw['id'].to_numpy()
ps_fstep_np     = ps_raw['future_step'].to_numpy()
ps_key_to_idx   = {
    (int(ps_ids_np[i]), int(ps_fstep_np[i])): i
    for i in range(len(ps_raw))
}

# ── ams_labeled (same join as in build_graph) ─────────────────────────────────
ams_raw = reachable_max_ships.filter(
    (pl.col('step_src') == T) & (pl.col('id_src').is_in(my_planet_ids))
)
act_flag = (
    actions_to_copy
    .filter(pl.col('step') == T)
    .select(['id_src', 'id_tgt'])
    .with_columns(pl.lit(1).alias('_match'))
)
ams_labeled = (
    ams_raw
    .join(act_flag, on=['id_src', 'id_tgt'], how='left')
    .with_columns(label=pl.col('_match').fill_null(0).cast(pl.Int64))
    .drop('_match')
)

# ── Validity filter: same as build_graph valid_ams_rows ───────────────────────
id_src_np   = ams_labeled['id_src'].to_numpy()
id_tgt_np   = ams_labeled['id_tgt'].to_numpy()
step_tgt_np = ams_labeled['step_tgt'].to_numpy()

valid_rows = [
    i for i in range(len(ams_labeled))
    if ps_key_to_idx.get((int(id_src_np[i]), T), -1) >= 0
    and ps_key_to_idx.get((int(id_tgt_np[i]), int(step_tgt_np[i])), -1) >= 0
]

assert len(valid_rows) == n_ams, (
    f'Mismatch: valid_rows={len(valid_rows)}, action_max nodes={n_ams}'
)

# ── Attach predictions ────────────────────────────────────────────────────────
result = (
    ams_labeled[valid_rows]
    .with_columns([
        pl.Series('prob',      probs.tolist()),
        pl.Series('predicted', (probs >= 0.5).tolist()),
    ])
    .sort(['id_src', 'id_tgt'])
)

print(f'action_max nodes: {n_ams}  positives: {n_pos}  predicted positives: {int((probs>=0.5).sum())}')
result.to_pandas()

action_max nodes: 39  positives: 1  predicted positives: 10


,id_src,step_src,angle,ships_sent,id_tgt,step_tgt,label,prob,predicted
0,12,35,-3.003457,2,1,54,0,0.058981,False
1,12,35,2.096969,2,5,49,0,0.874946,True
2,12,35,0.435150,2,14,54,0,0.951336,True
3,12,35,0.800887,2,18,51,0,0.928941,True
4,12,35,1.207544,2,20,38,0,0.989987,True
5,12,35,2.810567,2,25,49,0,0.057621,False
6,12,35,0.033505,2,30,54,0,0.083908,False
7,18,35,1.459978,1,0,46,0,0.879820,True
8,18,35,-0.048997,1,4,54,0,0.987658,True
9,18,35,-1.751253,1,14,40,1,0.991545,True


## Positives only — ground truth and predicted side by side

In [10]:
pos_gt   = result.filter(pl.col('label') == 1)
pos_pred = result.filter(pl.col('predicted') == True)

print('=== Ground truth positive actions ===')
display(pos_gt.select(['id_src','id_tgt','ships_sent','step_tgt','prob','label']).to_pandas())

print('=== Predicted positive actions (prob >= 0.5) ===')
display(pos_pred.select(['id_src','id_tgt','ships_sent','step_tgt','prob','label']).to_pandas())

# ── top-k by prob ─────────────────────────────────────────────────────────────
print('=== Top 10 by probability ===')
display(
    result
    .sort('prob', descending=True)
    .head(10)
    .select(['id_src','id_tgt','ships_sent','step_tgt','prob','label','predicted'])
    .to_pandas()
)

=== Ground truth positive actions ===


,id_src,id_tgt,ships_sent,step_tgt,prob,label
0,18,14,1,40,0.991545,1


=== Predicted positive actions (prob >= 0.5) ===


,id_src,id_tgt,ships_sent,step_tgt,prob,label
0,12,5,2,49,0.874946,0
1,12,14,2,54,0.951336,0
2,12,18,2,51,0.928941,0
3,12,20,2,38,0.989987,0
4,18,0,1,46,0.879820,0
5,18,4,1,54,0.987658,0
6,18,14,1,40,0.991545,1
7,18,22,1,42,0.991540,0
8,18,24,1,45,0.863965,0
9,22,23,11,49,0.567677,0


=== Top 10 by probability ===


,id_src,id_tgt,ships_sent,step_tgt,prob,label,predicted
0,18,14,1,40,0.991545,1,True
1,18,22,1,42,0.991540,0,True
2,12,20,2,38,0.989987,0,True
3,18,4,1,54,0.987658,0,True
4,12,14,2,54,0.951336,0,True
5,12,18,2,51,0.928941,0,True
6,18,0,1,46,0.879820,0,True
7,12,5,2,49,0.874946,0,True
8,18,24,1,45,0.863965,0,True
9,22,23,11,49,0.567677,0,True
